GNN Gaz — couplage réseau électrique via CCGTs
**Cibles** : injection des 4 puits gaz (nœuds 1, 5, 17, 18)
**Nouveauté** : `ccgt_demand[i,t]` = gaz consommé par les CCGTs raccordés au nœud i
+ `total_load[t]` et `total_wind[t]` comme contexte global

In [1]:
import os, json, random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


ModuleNotFoundError: No module named 'torch'

In [ ]:
@dataclass
class Config:
    # ── Gaz ───────────────────────────────────────────────────────────────────
    operating_point_path : str = "./output/Decision_variablesRU.xlsx"
    gas_network_path     : str = "./Gas Network Data.xlsx"
    gas_cons_ind_path    : str = "./input/Gas Cons Ind.xlsx"
    gas_cons_distrib_path: str = "./input/Gas Cons Distrib.xlsx"
    gas_cons_export_path : str = "./input/Gas Cons Export.xlsx"

    # ── Électricité ────────────────────────────────────────────────────────────
    elec_network_path: str = "./Elec Network Data.xlsx"
    elec_load_path   : str = "./input/Elec Load Data.xlsx"
    wind_data_path   : str = "./input/Wind Data.xlsx"

    output_dir: str = "outputs_gas_forecast"

    input_len : int = 72
    output_len: int = 24
    num_nodes : int = 28   # nœuds gaz
    num_pipes_expected: int = 33

    # Puits cibles (1-basés)
    target_nodes_1based: tuple = (1, 5, 17, 18)

    train_ratio: float = 0.70
    val_ratio  : float = 0.15
    test_ratio : float = 0.15

    hidden_dim     : int   = 128
    gnn_hidden_dim : int   = 128
    edge_hidden_dim: int   = 64
    dropout        : float = 0.10

    batch_size  : int   = 64
    lr          : float = 3e-4
    weight_decay: float = 1e-4
    max_epochs  : int   = 300
    patience    : int   = 30
    grad_clip   : float = 1.0

    use_huber  : bool  = False
    huber_delta: float = 0.5

    num_workers: int = 0
    seed       : int = 42

    # ── Couplage thermique ─────────────────────────────────────────────────────
    # Constante de conversion : P[MW] / η / GCV[kWh/Nm3] / 1000 = MNm3/h
    # GCV moyen belge ≈ 11.5 kWh/Nm3 (cohérent avec les fichiers Gas_Cons_*)
    GCV_kWh_per_Nm3: float = 11.5

    # ── Indices des features dans le vecteur par nœud ─────────────────────────
    # 0 : g_hist          (t, nœud) — non nul seulement aux 4 puits cibles
    # 1 : g_lag1          (t, nœud)
    # 2 : g_lag2          (t, nœud)
    # 3 : g_lag3          (t, nœud)
    # 4 : g_lag24         (t, nœud)
    # 5 : ind_global      (t seulement) — broadcast sur tous les nœuds
    # 6 : distrib_global  (t seulement) — broadcast
    # 7 : ccgt_demand     (t, nœud)    — NOUVEAU ; 0 si pas de CCGT sur ce nœud
    # 8 : total_load      (t seulement) — NOUVEAU ; broadcast
    # 9 : total_wind      (t seulement) — NOUVEAU ; broadcast
    # 10: res_share       (nœud)        — statique
    # 11: ind_share       (nœud)        — statique
    # 12: pr_min          (nœud)        — statique
    # 13: pr_max          (nœud)        — statique
    # 14: is_well         (nœud)        — statique
    # 15: is_gfpp         (nœud)        — statique
    # 16: is_export       (nœud)        — statique
    # 17: is_compressor   (nœud)        — statique
    # 18: sin_hour        (t)           — broadcast
    # 19: cos_hour        (t)           — broadcast
    # 20: sin_doy         (t)           — broadcast
    # 21: cos_doy         (t)           — broadcast
    # ─────────────────────────────────────────────────────────────────────────
    # total = 22 features / nœud / timestep
    #
    # context_vec (pour ContextGating) = features BROADCAST à t=-1 du nœud 0
    #   indices [5, 6, 8, 9] → context_dim = 4
    context_feature_indices: tuple = (5, 6, 8, 9)


CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)
CFG


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def clean_month_df(df: pd.DataFrame) -> pd.DataFrame:
    """Supprime les colonnes 'Unnamed: ...' générées par openpyxl."""
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
    return df.drop(columns=unnamed) if unnamed else df.copy()


def sorted_monthly_operating_sheets(path: str, prefix: str) -> list:
    """Retourne les noms de feuilles 'prefix-MM-YYYY' triées par mois."""
    xls = pd.ExcelFile(path)
    sheets = [s for s in xls.sheet_names if s.startswith(prefix)]
    return sorted(sheets, key=lambda s: int(s.split("-")[1]))


def sorted_monthly_generic_sheets(path: str) -> list:
    """Feuilles 'MM-YYYY', triées par mois, sans la feuille 'year'."""
    xls = pd.ExcelFile(path)
    sheets = [s for s in xls.sheet_names if s != "year"]
    return sorted(sheets, key=lambda s: int(s.split("-")[0]))


def read_monthly_operating_concat(path: str, prefix: str) -> pd.DataFrame:
    dfs = []
    for s in sorted_monthly_operating_sheets(path, prefix):
        df = clean_month_df(pd.read_excel(path, sheet_name=s))
        dfs.append(df.reset_index(drop=True))
    return pd.concat(dfs, axis=0, ignore_index=True)


def read_monthly_generic_concat(path: str) -> pd.DataFrame:
    dfs = []
    for s in sorted_monthly_generic_sheets(path):
        dfs.append(pd.read_excel(path, sheet_name=s).reset_index(drop=True))
    return pd.concat(dfs, axis=0, ignore_index=True)


def ensure_numeric(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    if out.isna().all().all():
        raise ValueError(f"{name}: aucune colonne numérique exploitable.")
    return out


def validate_shape(df: pd.DataFrame, expected_cols: int, name: str):
    if df.shape[1] != expected_cols:
        raise ValueError(
            f"{name}: colonnes attendues={expected_cols}, obtenu={df.shape[1]}"
        )


def impute_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
        if out[c].isna().any():
            med = out[c].median()
            out[c] = out[c].fillna(med if not np.isnan(med) else 0.0)
    return out


In [ ]:
def load_operating_point():
    """
    Charge g (injections 4 puits) depuis Decision_variablesRU.xlsx.
    Retourne : {"g_df": DataFrame[T×4], "datetime": Series[T]}
    """
    g_df = read_monthly_operating_concat(CFG.operating_point_path, "g-")
    g_df = ensure_numeric(g_df, "g")
    validate_shape(g_df, 4, "g")

    # Détecter la date de début depuis le nom de la première feuille (g-MM-YYYY)
    g_sheets = sorted_monthly_operating_sheets(CFG.operating_point_path, "g-")
    parts = g_sheets[0].split("-")          # ["g", "MM", "YYYY"]
    start_date = pd.Timestamp(year=int(parts[2]), month=int(parts[1]), day=1, hour=0)
    dt = pd.Series(pd.date_range(start=start_date, periods=len(g_df), freq="h"))

    print(f"Datetime : {dt.iloc[0]}  →  {dt.iloc[-1]}  ({len(dt)} heures)")
    return {"g_df": g_df, "datetime": dt}


op = load_operating_point()


In [ ]:
def load_exogenous(op: dict) -> dict:
    """
    Charge les consommations gaz agrégées (industrielle, distribution, export).
    Aligne sur la plage horaire de op.
    Retourne : {"ind_df", "distrib_df", "export_df"}  — chacun DataFrame[T×*]
    """
    ind_df     = read_monthly_generic_concat(CFG.gas_cons_ind_path)
    distrib_df = read_monthly_generic_concat(CFG.gas_cons_distrib_path)
    export_df  = read_monthly_generic_concat(CFG.gas_cons_export_path)

    for df in (ind_df, distrib_df, export_df):
        df["Date Time"] = pd.to_datetime(df["Date Time"])

    start, end = op["datetime"].iloc[0], op["datetime"].iloc[-1]
    ind_df     = ind_df    [(ind_df    ["Date Time"] >= start) & (ind_df    ["Date Time"] <= end)].reset_index(drop=True)
    distrib_df = distrib_df[(distrib_df["Date Time"] >= start) & (distrib_df["Date Time"] <= end)].reset_index(drop=True)
    export_df  = export_df [(export_df ["Date Time"] >= start) & (export_df ["Date Time"] <= end)].reset_index(drop=True)

    T = len(op["datetime"])
    assert len(ind_df) == len(distrib_df) == len(export_df) == T, \
        f"Longueurs exo incohérentes : ind={len(ind_df)}, distrib={len(distrib_df)}, export={len(export_df)}, T={T}"

    print(f"Exogènes OK  (T={T})")
    return {"ind_df": ind_df, "distrib_df": distrib_df, "export_df": export_df}


exo = load_exogenous(op)


In [ ]:
def load_network() -> dict:
    """
    Charge la topologie du réseau gaz depuis Gas_Network_Data.xlsx.
    Retourne : nodes_df, pipes_df, well_nodes, gfpp_nodes, export_nodes, compressor_nodes
    """
    nodes_df      = pd.read_excel(CFG.gas_network_path, sheet_name="Nodes")
    pipes_df      = pd.read_excel(CFG.gas_network_path, sheet_name="Pipelines")
    well_df       = pd.read_excel(CFG.gas_network_path, sheet_name="Well")
    gfpp_map_df   = pd.read_excel(CFG.gas_network_path, sheet_name="GFPPMap")

    pipes_df = impute_numeric_df(pipes_df[[
        "Pipeline", "From node", "To node",
        "Natural gas flow constant [MNm3/(bar*h)2]",
        "Compressor factor", "High Linepack",
        "Initial linepack [MNm3]", "Linepack constant [MNm3/bar]",
        "Estimated lenght [km]",
    ]])

    nodes_df = impute_numeric_df(nodes_df[[
        "Gas Node", "Res_Load_share_%", "Ind_Load_share_%",
        "Exp_Load_share_%", "Pr_min [bar]", "Pr_max [bar]",
    ]])

    well_nodes       = set(well_df["Gas node"].dropna().astype(int).tolist())
    gfpp_binary      = gfpp_map_df.fillna(0)
    # Ligne i (0-basé) = nœud gaz i+1 ; sum > 0 → au moins un CCGT
    gfpp_nodes       = set((gfpp_binary.sum(axis=1) > 0).index + 1)
    export_nodes     = {10, 21, 22, 25}
    comp_edges       = pipes_df.loc[pipes_df["Compressor factor"] > 1, ["From node", "To node"]]
    compressor_nodes = set(
        comp_edges["From node"].astype(int).tolist() +
        comp_edges["To node"].astype(int).tolist()
    )

    print(f"Nœuds puits      : {sorted(well_nodes)}")
    print(f"Nœuds GFPP       : {sorted(gfpp_nodes)}")
    print(f"Nœuds export     : {sorted(export_nodes)}")
    print(f"Nœuds compresseur: {sorted(compressor_nodes)}")
    return {
        "nodes_df": nodes_df, "pipes_df": pipes_df,
        "well_nodes": well_nodes, "gfpp_nodes": gfpp_nodes,
        "export_nodes": export_nodes, "compressor_nodes": compressor_nodes,
    }


net = load_network()


In [ ]:
def load_elec_coupling(op: dict) -> dict:
    """
    Calcule la demande gaz induite par les CCGTs à chaque nœud gaz, à chaque heure.

    Sources :
      - Decision_variablesRU.xlsx  feuilles p-MM-YYYY  → dispatch MW par générateur
      - Elec_Network_Data.xlsx     feuille Gen          → Gas node + rendement η

    Formule de conversion :
        consommation_gaz [MNm3/h] = P_j [MW] / (η_j × GCV [kWh/Nm3] × 1000)

    Les 4 puits cibles (1, 5, 17, 18) n'ont PAS de CCGT directement rattaché.
    Le signal se propage jusqu'à eux via la topologie des pipelines (message passing).

    Retourne :
        ccgt_demand : np.array [T × 28]  (MNm3/h ; 0 si pas de CCGT sur ce nœud)
        p_df        : pd.DataFrame [T × 24]  (dispatch MW brut)
                      colonne j (0-basé) = générateur j+1 (1-basé)
    """
    T = len(op["datetime"])

    # ── 1. Dispatch MW  (feuilles p-MM-YYYY) ──────────────────────────────────
    p_df = read_monthly_operating_concat(CFG.operating_point_path, "p-")
    p_df = ensure_numeric(p_df, "p")
    validate_shape(p_df, 24, "p")   # 24 générateurs après suppression Unnamed
    p_df = p_df.reset_index(drop=True)
    assert len(p_df) == T, f"p_df longueur {len(p_df)} ≠ T={T}"

    # ── 2. Mapping générateur → nœud gaz + rendement ──────────────────────────
    # Feuille Gen : colonne 'Generator' (1-basé), 'Gas node' (1-basé, NaN si nucléaire)
    gen_df = pd.read_excel(CFG.elec_network_path, sheet_name="Gen")
    ccgt_df = gen_df[gen_df["Gas node"].notna()].copy()
    ccgt_df["Gas node"]  = ccgt_df["Gas node"].astype(int)
    ccgt_df["Generator"] = ccgt_df["Generator"].astype(int)

    # ── 3. Calcul ccgt_demand [T × 28] ─────────────────────────────────────────
    ccgt_demand = np.zeros((T, CFG.num_nodes), dtype=np.float32)

    for _, row in ccgt_df.iterrows():
        gen_j      = int(row["Generator"])   # 1-basé → colonne p_df = gen_j - 1
        gas_node_i = int(row["Gas node"])     # 1-basé → index tableau  = gas_node_i - 1
        eta        = float(row["Efficiency"])

        p_col = p_df.iloc[:, gen_j - 1].to_numpy(dtype=np.float32)  # MW

        # MW  /  (η × GCV_kWh_Nm3 × 1000)  =  MNm3/h
        demand_MNm3 = p_col / (eta * CFG.GCV_kWh_per_Nm3 * 1000.0)
        ccgt_demand[:, gas_node_i - 1] += demand_MNm3

    # Résumé
    coupled_nodes = sorted(ccgt_df["Gas node"].unique())
    print(f"Nœuds gaz couplés (CCGT) : {coupled_nodes}")
    print(f"ccgt_demand — shape  : {ccgt_demand.shape}")
    print(f"cckt_demand — max    : {ccgt_demand.max():.4f} MNm3/h")
    print(f"ccgt_demand — nœuds non nuls : {(ccgt_demand.sum(0) > 0).sum()} / {CFG.num_nodes}")
    print(f"Puits cibles {list(CFG.target_nodes_1based)} — demande directe : "
          f"{[round(float(ccgt_demand[:, n-1].mean()), 5) for n in CFG.target_nodes_1based]}")

    return {"ccgt_demand": ccgt_demand, "p_df": p_df}


elec_coup = load_elec_coupling(op)


In [ ]:
def load_elec_context(op: dict) -> dict:
    """
    Charge la charge électrique totale et la production éolienne offshore (MW/h).
    Ces signaux sont broadcast à tous les nœuds gaz comme contexte global.

    Sources :
      - Elec_Load_Data.xlsx  feuille 'Year 2024'
      - Wind_Data.xlsx        feuille 'Year 2024'

    Les datetimes sont timezone-aware (+01:00 Europe/Brussels) → on strip la TZ.
    Alignement sur la plage de op["datetime"] (naive).
    """
    T = len(op["datetime"])

    def load_year_sheet(path: str, sheet: str, value_col: str) -> np.ndarray:
        df = pd.read_excel(path, sheet_name=sheet)
        # Strip timezone pour aligner avec op["datetime"] (naive)
        df["Datetime"] = pd.to_datetime(df["Datetime"]).dt.tz_localize(None)
        start, end = op["datetime"].iloc[0], op["datetime"].iloc[-1]
        df = df[(df["Datetime"] >= start) & (df["Datetime"] <= end)].reset_index(drop=True)
        assert len(df) == T, f"{path} | {sheet} : longueur {len(df)} ≠ T={T}"
        return df[value_col].to_numpy(dtype=np.float32)

    total_load = load_year_sheet(
        CFG.elec_load_path,
        sheet="Year 2024",
        value_col="Measured & Upscaled [MW]",
    )
    total_wind = load_year_sheet(
        CFG.wind_data_path,
        sheet="Year 2024",
        value_col="Measured & Upscaled [MW]",
    )

    print(f"total_load — shape={total_load.shape}  mean={total_load.mean():.1f} MW")
    print(f"total_wind — shape={total_wind.shape}  mean={total_wind.mean():.1f} MW")
    return {"total_load": total_load, "total_wind": total_wind}


elec_ctx = load_elec_context(op)


In [ ]:
T = len(op["datetime"])
assert len(op["g_df"])            == T
assert len(exo["ind_df"])         == T
assert len(exo["distrib_df"])     == T
assert len(exo["export_df"])      == T
assert elec_coup["ccgt_demand"].shape == (T, CFG.num_nodes)
assert elec_coup["p_df"].shape        == (T, 24)
assert elec_ctx["total_load"].shape   == (T,)
assert elec_ctx["total_wind"].shape   == (T,)

print(f"T = {T} heures  ✓")
print(f"Toutes les sources temporelles alignées  ✓")


In [ ]:
def build_graph(net: dict) -> dict:
    """
    Construit le graphe gaz bidirectionnel.
    Arêtes physiques (pipelines), attributs normalisés.
    """
    pipes_df = net["pipes_df"].copy()

    edge_feature_cols = [
        "Natural gas flow constant [MNm3/(bar*h)2]",
        "Compressor factor",
        "High Linepack",
        "Initial linepack [MNm3]",
        "Linepack constant [MNm3/bar]",
        "Estimated lenght [km]",
    ]
    edge_attr_df = impute_numeric_df(pipes_df[edge_feature_cols])

    edge_pairs, edge_attrs = [], []
    for i, row in pipes_df.iterrows():
        u = int(row["From node"]) - 1
        v = int(row["To node"]) - 1
        attr = edge_attr_df.iloc[i].to_numpy(dtype=np.float32)
        edge_pairs.append((u, v));  edge_attrs.append(attr)
        edge_pairs.append((v, u));  edge_attrs.append(attr)

    edge_index   = np.array(edge_pairs, dtype=np.int64).T
    edge_attr_raw = np.array(edge_attrs, dtype=np.float32)

    edge_scaler = StandardScaler()
    edge_attr   = edge_scaler.fit_transform(edge_attr_raw).astype(np.float32)

    nodes_df = net["nodes_df"].copy().sort_values("Gas Node").reset_index(drop=True)
    res_share = nodes_df["Res_Load_share_%"].to_numpy(dtype=np.float32)
    ind_share = nodes_df["Ind_Load_share_%"].to_numpy(dtype=np.float32)
    pr_min    = nodes_df["Pr_min [bar]"].to_numpy(dtype=np.float32)
    pr_max    = nodes_df["Pr_max [bar]"].to_numpy(dtype=np.float32)

    print(f"edge_index shape : {edge_index.shape}  "
          f"({edge_index.shape[1]//2} pipelines × 2 directions)")
    print(f"edge_attr  shape : {edge_attr.shape}")

    return {
        "edge_index": edge_index,
        "edge_attr" : edge_attr,
        "edge_attr_raw": edge_attr_raw,
        "edge_scaler"  : edge_scaler,
        "edge_feature_cols": edge_feature_cols,
        "res_share": res_share,
        "ind_share": ind_share,
        "pr_min"  : pr_min,
        "pr_max"  : pr_max,
    }


graph = build_graph(net)


In [ ]:
def build_node_features(
    op       : dict,
    exo      : dict,
    elec_coup: dict,
    elec_ctx : dict,
    net      : dict,
    graph    : dict,
) -> tuple:
    """
    Construit X_all_raw [T × N × 22] et Y_all_raw [T × 4].

    Features par nœud par timestep (22 au total) :
      0  g_hist          — injection au nœud (non nul aux 4 puits cibles)
      1  g_lag1
      2  g_lag2
      3  g_lag3
      4  g_lag24
      5  ind_global      — conso industrielle totale (broadcast)
      6  distrib_global  — conso distribuée totale   (broadcast)
      7  ccgt_demand     — gaz consommé par CCGTs à ce nœud [NOUVEAU]
      8  total_load      — charge élec totale         (broadcast) [NOUVEAU]
      9  total_wind      — éolien offshore total       (broadcast) [NOUVEAU]
     10  res_share       — fraction charge résidentielle (statique)
     11  ind_share       — fraction charge industrielle  (statique)
     12  pr_min          — pression min (statique)
     13  pr_max          — pression max (statique)
     14  is_well         — puits source (statique binaire)
     15  is_gfpp         — nœud GFPP    (statique binaire)
     16  is_export       — nœud export  (statique binaire)
     17  is_compressor   — nœud compresseur (statique binaire)
     18  sin_hour        — encodage temporel (broadcast)
     19  cos_hour
     20  sin_doy
     21  cos_doy
    """
    N = CFG.num_nodes
    T = len(op["datetime"])

    target_nodes  = list(CFG.target_nodes_1based)
    target_idx0   = [n - 1 for n in target_nodes]

    nodes_df  = net["nodes_df"].copy().sort_values("Gas Node").reset_index(drop=True)
    res_share = nodes_df["Res_Load_share_%"].to_numpy(dtype=np.float32)   # [N]
    ind_share = nodes_df["Ind_Load_share_%"].to_numpy(dtype=np.float32)   # [N]

    # ── 0-4 : historique g + lags ─────────────────────────────────────────────
    g_target    = op["g_df"].to_numpy(dtype=np.float32)          # [T, 4]
    g_hist_full = np.zeros((T, N), dtype=np.float32)
    for j, idx in enumerate(target_idx0):
        g_hist_full[:, idx] = g_target[:, j]

    g_lag1  = np.zeros((T, N), dtype=np.float32);  g_lag1 [1:]   = g_hist_full[:-1]
    g_lag2  = np.zeros((T, N), dtype=np.float32);  g_lag2 [2:]   = g_hist_full[:-2]
    g_lag3  = np.zeros((T, N), dtype=np.float32);  g_lag3 [3:]   = g_hist_full[:-3]
    g_lag24 = np.zeros((T, N), dtype=np.float32);  g_lag24[24:]  = g_hist_full[:-24]

    # ── 5-6 : consommations gaz globales (broadcast) ──────────────────────────
    ind_global     = exo["ind_df"]    ["Physical Flow (MNm3)"].to_numpy(dtype=np.float32)  # [T]
    distrib_global = exo["distrib_df"]["Physical Flow (MNm3)"].to_numpy(dtype=np.float32)  # [T]

    # ── 7 : ccgt_demand (par nœud — déjà [T × N]) ────────────────────────────
    ccgt_demand = elec_coup["ccgt_demand"]  # [T, N]

    # ── 8-9 : charge élec + éolien (broadcast) ────────────────────────────────
    total_load = elec_ctx["total_load"]  # [T]
    total_wind = elec_ctx["total_wind"]  # [T]

    # ── 10-17 : features statiques par nœud ───────────────────────────────────
    pr_min_arr = graph["pr_min"]       # [N]
    pr_max_arr = graph["pr_max"]       # [N]

    is_well       = np.array([1.0 if (i+1) in net["well_nodes"]       else 0.0 for i in range(N)], dtype=np.float32)
    is_gfpp       = np.array([1.0 if (i+1) in net["gfpp_nodes"]       else 0.0 for i in range(N)], dtype=np.float32)
    is_export     = np.array([1.0 if (i+1) in net["export_nodes"]     else 0.0 for i in range(N)], dtype=np.float32)
    is_compressor = np.array([1.0 if (i+1) in net["compressor_nodes"] else 0.0 for i in range(N)], dtype=np.float32)

    # ── 18-21 : encodage temporel ─────────────────────────────────────────────
    dt = op["datetime"]
    hour = dt.dt.hour.to_numpy(dtype=np.float32)
    doy  = dt.dt.dayofyear.to_numpy(dtype=np.float32)
    sin_hour = np.sin(2 * np.pi * hour / 24)
    cos_hour = np.cos(2 * np.pi * hour / 24)
    sin_doy  = np.sin(2 * np.pi * doy  / 365)
    cos_doy  = np.cos(2 * np.pi * doy  / 365)

    # ── Assemblage [T × N × 22] ───────────────────────────────────────────────
    def broadcast(arr_1d):
        """[T] → [T, N] en répétant sur les N nœuds."""
        return np.tile(arr_1d[:, None], (1, N)).astype(np.float32)

    X_all_raw = np.stack([
        g_hist_full,                  # 0
        g_lag1,                        # 1
        g_lag2,                        # 2
        g_lag3,                        # 3
        g_lag24,                       # 4
        broadcast(ind_global),         # 5
        broadcast(distrib_global),     # 6
        ccgt_demand,                   # 7  [T, N]
        broadcast(total_load),         # 8
        broadcast(total_wind),         # 9
        np.tile(res_share,       (T, 1)),  # 10
        np.tile(ind_share,       (T, 1)),  # 11
        np.tile(pr_min_arr,      (T, 1)),  # 12
        np.tile(pr_max_arr,      (T, 1)),  # 13
        np.tile(is_well,         (T, 1)),  # 14
        np.tile(is_gfpp,         (T, 1)),  # 15
        np.tile(is_export,       (T, 1)),  # 16
        np.tile(is_compressor,   (T, 1)),  # 17
        broadcast(sin_hour),           # 18
        broadcast(cos_hour),           # 19
        broadcast(sin_doy),            # 20
        broadcast(cos_doy),            # 21
    ], axis=-1)   # → [T, N, 22]

    Y_all_raw = g_target.copy()  # [T, 4]

    feature_names = [
        "g_hist", "g_lag1", "g_lag2", "g_lag3", "g_lag24",
        "ind_global", "distrib_global", "ccgt_demand",
        "total_load", "total_wind",
        "res_share", "ind_share", "pr_min", "pr_max",
        "is_well", "is_gfpp", "is_export", "is_compressor",
        "sin_hour", "cos_hour", "sin_doy", "cos_doy",
    ]

    print(f"X_all_raw : {X_all_raw.shape}  (T={T}, N={N}, F={X_all_raw.shape[-1]})")
    print(f"Y_all_raw : {Y_all_raw.shape}")
    print(f"ccgt_demand non nul aux nœuds : "
          f"{[i+1 for i in range(N) if ccgt_demand[:,i].sum() > 0]}")

    return X_all_raw, Y_all_raw, {
        "feature_names": feature_names,
        "target_idx0"  : target_idx0,
    }


X_all_raw, Y_all_raw, feat = build_node_features(op, exo, elec_coup, elec_ctx, net, graph)


In [ ]:
def compute_boundaries(T: int):
    train_end = int(T * CFG.train_ratio)
    val_end   = int(T * (CFG.train_ratio + CFG.val_ratio))
    return train_end, val_end


train_end, val_end = compute_boundaries(T)
print(f"train_end={train_end}  val_end={val_end}  T={T}")


In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 30):
        self.patience = patience
        self.best     = float("inf")
        self.counter  = 0

    def step(self, val_loss: float):
        improved = val_loss < self.best
        if improved:
            self.best    = val_loss
            self.counter = 0
        else:
            self.counter += 1
        return improved, self.counter >= self.patience


In [ ]:
def fit_scalers(X_raw: np.ndarray, Y_raw: np.ndarray, train_end: int):
    """Un StandardScaler global pour X, un par nœud cible pour Y."""
    Fdim = X_raw.shape[-1]
    x_scaler = StandardScaler()
    x_scaler.fit(X_raw[:train_end].reshape(-1, Fdim))

    y_scalers = []
    for i in range(Y_raw.shape[-1]):
        sc = StandardScaler()
        sc.fit(Y_raw[:train_end, i:i+1])
        y_scalers.append(sc)

    return x_scaler, y_scalers


def apply_scalers(X_raw, Y_raw, x_scaler, y_scalers):
    T, N, Fdim = X_raw.shape
    X_scaled = x_scaler.transform(
        X_raw.reshape(-1, Fdim)
    ).reshape(T, N, Fdim).astype(np.float32)

    Y_scaled = np.zeros_like(Y_raw, dtype=np.float32)
    for i, sc in enumerate(y_scalers):
        Y_scaled[:, i:i+1] = sc.transform(Y_raw[:, i:i+1])

    return X_scaled, Y_scaled


def inverse_transform_y(y_scaled: np.ndarray, y_scalers: list) -> np.ndarray:
    sh = y_scaled.shape
    y_out = np.empty_like(y_scaled, dtype=float)
    for j, sc in enumerate(y_scalers):
        col = y_scaled[..., j].reshape(-1, 1)
        y_out[..., j] = sc.inverse_transform(col).reshape(sh[:-1])
    return y_out


x_scaler, y_scalers = fit_scalers(X_all_raw, Y_all_raw, train_end)
X_all, Y_all        = apply_scalers(X_all_raw, Y_all_raw, x_scaler, y_scalers)

print(f"X_all : {X_all.shape}   Y_all : {Y_all.shape}")
for i, sc in enumerate(y_scalers):
    print(f"  Puits {CFG.target_nodes_1based[i]} → mean={sc.mean_[0]:.4f}, std={sc.scale_[0]:.4f}")


In [ ]:
def build_windows(X, Y, datetimes, input_len, output_len, train_end, val_end):
    Xs, Ys, date_out, splits = [], [], [], []
    max_start = len(X) - input_len - output_len + 1

    for s in range(max_start):
        in_s, in_e   = s, s + input_len
        out_s, out_e = in_e, in_e + output_len

        Xs.append(X[in_s:in_e])
        Ys.append(Y[out_s:out_e])
        date_out.append(datetimes.iloc[out_s:out_e].to_numpy())

        if out_e <= train_end:
            splits.append("train")
        elif out_e <= val_end:
            splits.append("val")
        else:
            splits.append("test")

    Xs    = np.stack(Xs).astype(np.float32)
    Ys    = np.stack(Ys).astype(np.float32)
    date_out = np.array(date_out, dtype="datetime64[ns]")
    splits   = np.array(splits)
    return Xs, Ys, date_out, splits


X_seq, Y_seq, date_seq, split_labels = build_windows(
    X_all, Y_all, op["datetime"],
    CFG.input_len, CFG.output_len, train_end, val_end,
)
print(f"X_seq={X_seq.shape}  Y_seq={Y_seq.shape}")
print(f"train={np.sum(split_labels=='train')}  "
      f"val={np.sum(split_labels=='val')}  "
      f"test={np.sum(split_labels=='test')}")


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], idx


def make_loader(split_name: str):
    mask = split_labels == split_name
    ds   = SeqDataset(X_seq[mask], Y_seq[mask])
    loader = DataLoader(
        ds,
        batch_size=CFG.batch_size,
        shuffle=(split_name == "train"),
        num_workers=CFG.num_workers,
    )
    return ds, loader


train_ds, train_loader = make_loader("train")
val_ds,   val_loader   = make_loader("val")
test_ds,  test_loader  = make_loader("test")
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")


In [ ]:
class EdgeAwareGraphConv(nn.Module):
    """Convolution de graphe pondérée par les attributs physiques des arêtes."""

    def __init__(self, in_dim, out_dim, edge_dim, edge_hidden_dim=16, dropout=0.0):
        super().__init__()
        self.msg_lin  = nn.Linear(in_dim, out_dim, bias=False)
        self.self_lin = nn.Linear(in_dim, out_dim, bias=True)
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim, edge_hidden_dim),
            nn.ReLU(),
            nn.Linear(edge_hidden_dim, 1),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        # x          : [B, N, in_dim]
        # edge_index : [2, E]
        # edge_attr  : [E, edge_dim]
        B, N, _ = x.shape
        src, dst = edge_index[0], edge_index[1]

        msg = self.msg_lin(x[:, src, :])                    # [B, E, out_dim]
        w   = torch.sigmoid(self.edge_mlp(edge_attr))       # [E, 1]
        msg = msg * w.unsqueeze(0)                           # [B, E, out_dim]

        out = self.self_lin(x)                               # [B, N, out_dim]
        agg = torch.zeros(B, N, msg.shape[-1], device=x.device)
        idx = dst.view(1, -1, 1).expand(B, -1, msg.shape[-1])
        agg.scatter_add_(1, idx, msg)

        deg = torch.zeros(N, device=x.device)
        deg.scatter_add_(0, dst, torch.ones(dst.shape[0], device=x.device))
        agg = agg / deg.clamp_min(1.0).view(1, N, 1)

        return self.dropout(F.relu(out + agg))


class AdaptiveGraphConv(nn.Module):
    """
    Adjacence apprise (Graph WaveNet) : A = softmax(ReLU(E · Eᵀ)).
    Capture des dépendances cachées indépendamment de la topologie physique.
    """

    def __init__(self, in_dim, out_dim, n_nodes, emb_dim=10, dropout=0.0):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, emb_dim)
        self.msg_lin  = nn.Linear(in_dim, out_dim, bias=False)
        self.self_lin = nn.Linear(in_dim, out_dim, bias=True)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x):
        # x : [B, N, in_dim]
        E = self.node_emb.weight                               # [N, emb_dim]
        A = F.softmax(F.relu(E @ E.t()), dim=-1)              # [N, N]
        msg = self.msg_lin(x)                                  # [B, N, out_dim]
        agg = torch.einsum("ij,bjd->bid", A, msg)              # [B, N, out_dim]
        return self.dropout(F.relu(self.self_lin(x) + agg))


In [ ]:
class ContextGating(nn.Module):
    """
    Module le représentation des puits cibles par un vecteur de contexte global.
    context_dim = 4  (ind_global, distrib_global, total_load, total_wind)
    """

    def __init__(self, hidden_dim, context_dim, dropout=0.1):
        super().__init__()
        self.context_proj = nn.Sequential(
            nn.Linear(context_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid(),
        )
        self.context_transform = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, h_target, context_vec):
        # h_target   : [B, n_wells, hidden_dim]
        # context_vec: [B, context_dim]
        B, n_tgt, D = h_target.shape
        ctx  = self.context_proj(context_vec)                  # [B, hidden_dim]
        ctx  = ctx.unsqueeze(1).expand(-1, n_tgt, -1)          # [B, n_tgt, hidden_dim]
        gate = self.gate(torch.cat([h_target, ctx], dim=-1))   # [B, n_tgt, hidden_dim]
        corr = self.context_transform(ctx)
        return h_target + gate * corr


class STGasForecasterContextGated(nn.Module):
    """
    Prédicteur spatio-temporel sur le sous-graphe puits + voisins directs.

    Architecture :
      1. MLP temporel   : aplatit la fenêtre (T_in × F) par nœud → hidden_dim
      2. EdgeAwareConv  : message passing sur arêtes physiques (sous-graphe 8 nœuds)
      3. AdaptiveConv   : adjacence apprise sur le sous-graphe
      4. ContextGating  : modulation des 4 puits par le contexte global électrique+gaz
      5. Décodeur MLP   : hidden_dim → output_len par puits

    context_vec est extrait automatiquement depuis l'entrée :
      features broadcast [5, 6, 8, 9] au dernier timestep, nœud 0.
    """

    def __init__(
        self,
        input_dim,
        edge_dim,
        hidden_dim,
        gnn_hidden_dim,
        output_len,
        target_node_idx0,
        input_len,
        num_nodes     = 28,
        n_sub         = 8,
        well_local    = None,
        dropout       = 0.1,
        context_dim   = 4,
        context_feat_indices = (5, 6, 8, 9),
    ):
        super().__init__()
        self.target_node_idx0     = target_node_idx0
        self.output_len           = output_len
        self.n_sub                = n_sub
        self.well_local           = well_local if well_local is not None else [0, 2, 4, 6]
        self.num_nodes            = num_nodes
        self.context_feat_indices = list(context_feat_indices)

        # Encodeur temporel par nœud : T_in × F → hidden_dim
        self.temporal_mlp = nn.Sequential(
            nn.Linear(input_len * input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Couches GNN
        self.gnn1 = EdgeAwareGraphConv(
            hidden_dim, gnn_hidden_dim, edge_dim,
            edge_hidden_dim=CFG.edge_hidden_dim, dropout=dropout,
        )
        self.adapt_conv = AdaptiveGraphConv(
            gnn_hidden_dim, gnn_hidden_dim,
            n_nodes=n_sub, emb_dim=10, dropout=dropout,
        )

        # Context gating
        self.context_gate = ContextGating(gnn_hidden_dim, context_dim, dropout)

        # Décodeur : un par puits cible
        n_targets = len(target_node_idx0)
        self.decoder = nn.Sequential(
            nn.Linear(gnn_hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_len),
        )

    def forward(self, x, edge_index_sub, edge_attr_sub, sub_nodes):
        """
        x              : [B, T_in, N_full, F]
        edge_index_sub : [2, E_sub]
        edge_attr_sub  : [E_sub, edge_dim]
        sub_nodes      : list of n_sub global node indices
        """
        B, T, N, F = x.shape

        # ── Context global (features broadcast au dernier pas de temps) ────────
        # features [5,6,8,9] identiques sur tous les nœuds → on lit le nœud 0
        context_vec = x[:, -1, 0, self.context_feat_indices]   # [B, context_dim]

        # ── Sous-graphe des 8 nœuds ──────────────────────────────────────────
        x_sub  = x[:, :, sub_nodes, :]          # [B, T, n_sub, F]
        x_flat = x_sub.reshape(B, self.n_sub, T * F)   # [B, n_sub, T*F]

        # ── Encodeur temporel ─────────────────────────────────────────────────
        h = self.temporal_mlp(x_flat)           # [B, n_sub, hidden_dim]

        # ── GNN ───────────────────────────────────────────────────────────────
        h = self.gnn1(h, edge_index_sub, edge_attr_sub)  # [B, n_sub, gnn_hidden_dim]
        h = self.adapt_conv(h)                            # [B, n_sub, gnn_hidden_dim]

        # ── Extraire les 4 puits (well_local = positions dans le sous-graphe) ─
        h_wells = h[:, self.well_local, :]       # [B, 4, gnn_hidden_dim]

        # ── Context gating ────────────────────────────────────────────────────
        h_wells = self.context_gate(h_wells, context_vec)  # [B, 4, gnn_hidden_dim]

        # ── Décodage ─────────────────────────────────────────────────────────
        out = self.decoder(h_wells)              # [B, 4, output_len]
        return out.permute(0, 2, 1)              # [B, output_len, 4]


In [ ]:
class MLPBaseline(nn.Module):
    """Baseline MLP pur : aplatit tous les nœuds cibles × temps → prédit 24h."""

    def __init__(self, input_dim, hidden_dim, output_len, target_node_idx0, input_len, dropout=0.1):
        super().__init__()
        n_targets  = len(target_node_idx0)
        in_features = input_len * n_targets * input_dim
        self.target_node_idx0 = target_node_idx0

        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_len * n_targets),
        )
        self.output_len = output_len
        self.n_targets  = n_targets

    def forward(self, x, *args):
        B, T, N, F = x.shape
        x_target = x[:, :, self.target_node_idx0, :]     # [B, T, 4, F]
        x_flat   = x_target.reshape(B, -1)               # [B, T*4*F]
        out      = self.net(x_flat)                       # [B, output_len * 4]
        return out.reshape(B, self.output_len, self.n_targets)


In [ ]:
def build_well_neighbor_subgraph(graph: dict, feat: dict, net: dict) -> dict:
    """
    Sous-graphe de 8 nœuds : 4 puits + 4 voisins directs (GFPP/compresseurs).

    Topologie (d'après Pipelines) :
      Nœud 1  (idx 0)  ↔ Nœud 2  (idx 1)
      Nœud 5  (idx 4)  ↔ Nœud 6  (idx 5)
      Nœud 17 (idx 16) ↔ Nœud 16 (idx 15)
      Nœud 18 (idx 17) ↔ Nœud 19 (idx 18)

    Indices locaux dans le sous-graphe :
      0=Nœud1  1=Nœud2  2=Nœud5  3=Nœud6
      4=Nœud17 5=Nœud16 6=Nœud18 7=Nœud19
    """
    global_nodes = [0, 1, 4, 5, 16, 15, 17, 18]   # 0-basés
    global_to_local = {g: l for l, g in enumerate(global_nodes)}

    well_neighbor_pairs = [
        (0,  1),    # Nœud 1  ↔ Nœud 2
        (4,  5),    # Nœud 5  ↔ Nœud 6
        (16, 15),   # Nœud 17 ↔ Nœud 16
        (17, 18),   # Nœud 18 ↔ Nœud 19
    ]

    edge_index_full = graph["edge_index"]
    edge_attr_full  = graph["edge_attr"]
    phys_edges = {
        (int(edge_index_full[0, e]), int(edge_index_full[1, e])): edge_attr_full[e]
        for e in range(edge_index_full.shape[1])
    }

    sub_edges, sub_attrs = [], []
    for u, v in well_neighbor_pairs:
        for a, b in [(u, v), (v, u)]:
            if (a, b) in phys_edges:
                sub_edges.append((global_to_local[a], global_to_local[b]))
                sub_attrs.append(phys_edges[(a, b)])

    edge_index_sub = np.array(sub_edges, dtype=np.int64).T
    edge_attr_sub  = np.stack(sub_attrs).astype(np.float32)

    # well_local : positions des 4 puits dans le sous-graphe
    well_local = [global_to_local[g] for g in feat["target_idx0"]]

    print(f"Sous-graphe : {len(global_nodes)} nœuds  |  {edge_index_sub.shape[1]} arêtes")
    print(f"  Nœuds globaux : {[n+1 for n in global_nodes]}")
    print(f"  well_local    : {well_local}  (positions des puits dans le sous-graphe)")

    return {
        "sub_nodes"      : global_nodes,
        "n_sub"          : len(global_nodes),
        "well_local"     : well_local,
        "edge_index_sub" : edge_index_sub,
        "edge_attr_sub"  : edge_attr_sub,
    }


subgraph = build_well_neighbor_subgraph(graph, feat, net)

edge_index_sub_t = torch.tensor(subgraph["edge_index_sub"], dtype=torch.long,    device=device)
edge_attr_sub_t  = torch.tensor(subgraph["edge_attr_sub"],  dtype=torch.float32, device=device)


In [ ]:
def run_epoch_graph(model, loader, optimizer=None):
    is_train  = optimizer is not None
    model.train(is_train)
    criterion = nn.HuberLoss(delta=CFG.huber_delta) if CFG.use_huber else nn.MSELoss()

    total_loss, preds, trues = 0.0, [], []

    with torch.set_grad_enabled(is_train):
        for xb, yb, _ in loader:
            xb, yb = xb.to(device), yb.to(device)
            if is_train:
                optimizer.zero_grad()

            yhat = model(xb, edge_index_sub_t, edge_attr_sub_t, subgraph["sub_nodes"])
            loss = criterion(yhat, yb)

            if is_train:
                loss.backward()
                if CFG.grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds.append(yhat.detach().cpu().numpy())
            trues.append(yb.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.concatenate(preds), np.concatenate(trues)


def run_epoch_baseline(model, loader, optimizer=None):
    is_train  = optimizer is not None
    model.train(is_train)
    criterion = nn.HuberLoss(delta=CFG.huber_delta) if CFG.use_huber else nn.MSELoss()

    total_loss, preds, trues = 0.0, [], []

    with torch.set_grad_enabled(is_train):
        for xb, yb, _ in loader:
            xb, yb = xb.to(device), yb.to(device)
            if is_train:
                optimizer.zero_grad()

            yhat = model(xb)
            loss = criterion(yhat, yb)

            if is_train:
                loss.backward()
                if CFG.grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds.append(yhat.detach().cpu().numpy())
            trues.append(yb.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.concatenate(preds), np.concatenate(trues)


In [ ]:
def fit_model(model, model_name="model", graph_model=True):
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=8
    )
    early    = EarlyStopping(patience=CFG.patience)
    history  = {"train_loss": [], "val_loss": []}
    best_path = os.path.join(CFG.output_dir, f"best_{model_name}.pt")

    run_fn = run_epoch_graph if graph_model else run_epoch_baseline

    for epoch in range(1, CFG.max_epochs + 1):
        train_loss, _, _ = run_fn(model, train_loader, optimizer)
        val_loss,   _, _ = run_fn(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        scheduler.step(val_loss)

        improved, stop = early.step(val_loss)
        if improved:
            torch.save(model.state_dict(), best_path)

        print(f"Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f} "
              f"| best={early.best:.6f} | lr={optimizer.param_groups[0]['lr']:.2e}")
        if stop:
            print("Early stopping.")
            break

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, history, best_path


def predict_dataset(model, loader, graph_model=True):
    run_fn = run_epoch_graph if graph_model else run_epoch_baseline
    _, preds_scaled, trues_scaled = run_fn(model, loader)
    preds = inverse_transform_y(preds_scaled, y_scalers)
    trues = inverse_transform_y(trues_scaled, y_scalers)
    return preds, trues


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Paramètres entraînables : {train:,} / {total:,}")
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(f"  {name:<45s}  {str(list(p.shape)):<20s}  {p.numel():>8,}")
    return train


In [ ]:
def compute_metrics(y_true, y_pred):
    err  = y_pred - y_true
    mae  = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mape = float(np.mean(np.abs(err) / np.clip(np.abs(y_true), 1e-6, None)) * 100)

    mean_abs = float(np.mean(np.abs(y_true)))
    mae_over_mean  = mae  / mean_abs if mean_abs > 0 else np.nan
    rmse_over_mean = rmse / mean_abs if mean_abs > 0 else np.nan

    rmse_per_node    = np.sqrt(np.mean(err ** 2, axis=(0, 1))).tolist()
    rmse_per_horizon = np.sqrt(np.mean(err ** 2, axis=(0, 2))).tolist()
    heatmap_rmse     = np.sqrt(np.mean(err ** 2, axis=0)).T.tolist()

    return {
        "MAE" : mae,  "RMSE": rmse,  "MAPE": mape,
        "MAE/MEAN": mae_over_mean,
        "RMSE/MEAN": rmse_over_mean,
        "RMSE_per_node"   : rmse_per_node,
        "RMSE_per_horizon": rmse_per_horizon,
        "heatmap_rmse"    : heatmap_rmse,
    }


In [ ]:
input_dim = X_seq.shape[-1]   # 22
edge_dim  = subgraph["edge_attr_sub"].shape[-1]
target_idx0 = feat["target_idx0"]
context_dim = len(CFG.context_feature_indices)  # 4

print(f"input_dim   = {input_dim}  (22 features)")
print(f"edge_dim    = {edge_dim}")
print(f"context_dim = {context_dim}  (ind, distrib, load, wind)")
print(f"target_idx0 = {target_idx0}")

graph_model = STGasForecasterContextGated(
    input_dim             = input_dim,
    edge_dim              = edge_dim,
    hidden_dim            = CFG.hidden_dim,
    gnn_hidden_dim        = CFG.gnn_hidden_dim,
    output_len            = CFG.output_len,
    target_node_idx0      = target_idx0,
    input_len             = CFG.input_len,
    num_nodes             = CFG.num_nodes,
    n_sub                 = subgraph["n_sub"],
    well_local            = subgraph["well_local"],
    dropout               = CFG.dropout,
    context_dim           = context_dim,
    context_feat_indices  = CFG.context_feature_indices,
).to(device)

baseline_model = MLPBaseline(
    input_dim        = input_dim,
    hidden_dim       = CFG.hidden_dim,
    output_len       = CFG.output_len,
    target_node_idx0 = target_idx0,
    input_len        = CFG.input_len,
    dropout          = CFG.dropout,
).to(device)

print("\n=== GRAPH MODEL ===")
count_parameters(graph_model)
print("\n=== BASELINE ===")
count_parameters(baseline_model)


In [ ]:
print("=== Training graph model ===")
graph_model, hist_graph, path_graph = fit_model(
    graph_model, model_name="graph_stgnn_elec", graph_model=True,
)

print("\n=== Training baseline ===")
baseline_model, hist_base, path_base = fit_model(
    baseline_model, model_name="baseline_mlp", graph_model=False,
)


In [ ]:
graph_pred, graph_true = predict_dataset(graph_model,    test_loader, graph_model=True)
base_pred,  base_true  = predict_dataset(baseline_model, test_loader, graph_model=False)

metrics_graph = compute_metrics(graph_true, graph_pred)
metrics_base  = compute_metrics(base_true,  base_pred)

print("=== GRAPH MODEL ===")
for k in ["MAE", "RMSE", "MAPE", "MAE/MEAN", "RMSE/MEAN"]:
    print(f"  {k:12s} = {metrics_graph[k]:.4f}")
print(f"  RMSE/nœud  = {metrics_graph['RMSE_per_node']}")

print("\n=== BASELINE ===")
for k in ["MAE", "RMSE", "MAPE", "MAE/MEAN", "RMSE/MEAN"]:
    print(f"  {k:12s} = {metrics_base[k]:.4f}")
print(f"  RMSE/nœud  = {metrics_base['RMSE_per_node']}")

# ── Sauvegardes ───────────────────────────────────────────────────────────────
def to_serializable(m):
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in m.items()}

save_json(to_serializable(metrics_graph), os.path.join(CFG.output_dir, "metrics_graph.json"))
save_json(to_serializable(metrics_base),  os.path.join(CFG.output_dir, "metrics_baseline.json"))

np.save(os.path.join(CFG.output_dir, "graph_pred.npy"),    graph_pred)
np.save(os.path.join(CFG.output_dir, "graph_true.npy"),    graph_true)
np.save(os.path.join(CFG.output_dir, "baseline_pred.npy"), base_pred)
np.save(os.path.join(CFG.output_dir, "baseline_true.npy"), base_true)

save_json(
    {"mean": x_scaler.mean_.tolist(), "scale": x_scaler.scale_.tolist()},
    os.path.join(CFG.output_dir, "x_scaler.json"),
)
save_json(
    {"mean":  [sc.mean_.tolist()  for sc in y_scalers],
     "scale": [sc.scale_.tolist() for sc in y_scalers]},
    os.path.join(CFG.output_dir, "y_scalers.json"),
)
save_json(
    {"mean":          graph["edge_scaler"].mean_.tolist(),
     "scale":         graph["edge_scaler"].scale_.tolist(),
     "feature_names": graph["edge_feature_cols"]},
    os.path.join(CFG.output_dir, "edge_scaler.json"),
)
print("\nFichiers sauvegardés dans :", CFG.output_dir)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_graph["train_loss"], label="Graph train")
ax.plot(hist_graph["val_loss"],   label="Graph val")
ax.plot(hist_base["train_loss"],  label="Baseline train", linestyle="--")
ax.plot(hist_base["val_loss"],    label="Baseline val",   linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Courbes d'entraînement")
plt.tight_layout(); plt.show()


In [ ]:
node_labels = [str(n) for n in CFG.target_nodes_1based]
x = np.arange(len(node_labels)); w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, metrics_graph["RMSE_per_node"], width=w, label="Graph")
ax.bar(x + w/2, metrics_base["RMSE_per_node"],  width=w, label="Baseline")
ax.set_xticks(x); ax.set_xticklabels(node_labels)
ax.set_xlabel("Puits cible (nœud gaz)"); ax.set_ylabel("RMSE"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("RMSE par puits cible")
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, CFG.output_len+1), metrics_graph["RMSE_per_horizon"], marker="o", label="Graph")
ax.plot(range(1, CFG.output_len+1), metrics_base["RMSE_per_horizon"],  marker="s", label="Baseline")
ax.set_xlabel("Horizon (h)"); ax.set_ylabel("RMSE"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("RMSE par horizon de prédiction")
plt.tight_layout(); plt.show()


In [ ]:
num_plot = min(72, len(graph_pred))
node_labels = [f"Puits {n}" for n in CFG.target_nodes_1based]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(graph_true[:num_plot, 0, i], label="Réel")
    ax.plot(graph_pred[:num_plot, 0, i], label="Graph")
    ax.plot(base_pred [:num_plot, 0, i], label="Baseline", linestyle="--", alpha=0.7)
    ax.set_ylabel(node_labels[i]); ax.grid(alpha=0.3)
axes[0].legend(ncol=3, fontsize=9)
axes[-1].set_xlabel("Échantillons test")
fig.suptitle("Prédiction h+1 sur le test set")
plt.tight_layout(); plt.show()


In [ ]:
hm = np.array(metrics_graph["heatmap_rmse"])   # [4, 24]
fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(hm, aspect="auto")
plt.colorbar(im, ax=ax, label="RMSE")
ax.set_yticks(range(4)); ax.set_yticklabels(node_labels)
ax.set_xticks(range(CFG.output_len))
ax.set_xticklabels(range(1, CFG.output_len+1), rotation=90)
ax.set_xlabel("Horizon (h)"); ax.set_ylabel("Puits")
ax.set_title("Heatmap RMSE [puits × horizon] — Graph model")
plt.tight_layout(); plt.show()


In [ ]:
# Visualise la demande CCGT agrégée sur les nœuds couplés vs injection des puits
dt_series = pd.to_datetime(op["datetime"])

ccgt_total = elec_coup["ccgt_demand"].sum(axis=1)   # somme sur 28 nœuds
g_total    = op["g_df"].to_numpy().sum(axis=1)       # somme des 4 puits

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(dt_series, g_total,    color="steelblue", linewidth=0.6, label="Σ injection puits")
axes[0].set_ylabel("MNm3/h"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(dt_series, ccgt_total, color="coral",     linewidth=0.6, label="Σ demande CCGT")
axes[1].set_ylabel("MNm3/h"); axes[1].legend(); axes[1].grid(alpha=0.3)

r = np.corrcoef(g_total, ccgt_total)[0, 1]
fig.suptitle(f"Injection puits vs demande thermique CCGT  (r={r:.3f})")
plt.tight_layout(); plt.show()
print(f"Corrélation Σpuits ↔ Σdemande_CCGT : {r:.4f}")
